# 🌊 Flood Risk Prediction
### Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

print('Libraries imported!')

### Step 2 — Load Dataset

In [ ]:
# Make sure train.csv is in the same folder as this notebook
df = pd.read_csv('train.csv')

print('Dataset loaded!')
print('Shape:', df.shape)
print()
print(df.head())

### Step 3 — Explore the Data

In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Missing values:', df.isnull().sum().sum())
print()
print('FloodProbability range:')
print('Min:', df['FloodProbability'].min())
print('Max:', df['FloodProbability'].max())
print('Mean:', round(df['FloodProbability'].mean(), 3))

### Step 4 — Convert Probability to Risk Category
Target is a float (0.0 to 1.0) so we convert it to 3 classes

In [ ]:
# Convert flood probability to risk level
# Low    = probability < 0.40
# Medium = probability 0.40 to 0.60
# High   = probability > 0.60

def get_risk(prob):
    if prob < 0.40:
        return 'Low'
    elif prob <= 0.60:
        return 'Medium'
    else:
        return 'High'

df['RiskLevel'] = df['FloodProbability'].apply(get_risk)

print('Risk Level Distribution:')
print(df['RiskLevel'].value_counts())

### Step 5 — Prepare Features and Target

In [ ]:
# Drop id and original probability column
# Use all 20 features
feature_cols = [
    'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
    'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
    'Siltation', 'AgriculturalPractices', 'Encroachments',
    'IneffectiveDisasterPreparedness', 'DrainageSystems',
    'CoastalVulnerability', 'Landslides', 'Watersheds',
    'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
    'InadequatePlanning', 'PoliticalFactors'
]

X = df[feature_cols]
y = df['RiskLevel']

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Features shape:', X.shape)
print('Classes:', le.classes_)  # High=0, Low=1, Medium=2
print()
print('X sample:')
print(X.head())

### Step 6 — Split Data
Dataset is very large (1.1M rows) so we use only 100,000 rows for faster training

In [ ]:
# Use 100,000 rows — enough for great accuracy, much faster training
X_sample = X.sample(n=100000, random_state=42)
y_sample = y_encoded[X_sample.index]

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42
)

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)

### Step 7 — Train Random Forest Model

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print('Model trained!')

### Step 8 — Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

### Step 9 — Export Model as .pkl

In [ ]:
# Save trained model
with open('flood_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save label encoder
with open('flood_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save feature column names (important for correct order in app)
with open('flood_features.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print('flood_model.pkl   --> saved!')
print('flood_encoder.pkl --> saved!')
print('flood_features.pkl --> saved!')

### Step 10 — Test the Saved Model

In [ ]:
# Load model back
with open('flood_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('flood_encoder.pkl', 'rb') as f:
    loaded_le = pickle.load(f)

# Test sample — high risk values
# All features rated 8-10 out of 10 = high flood risk
sample = [[9, 8, 8, 9, 8, 9, 8, 8, 8, 9, 9, 8, 9, 8, 8, 9, 8, 9, 9, 8]]
pred  = loaded_model.predict(sample)
label = loaded_le.inverse_transform(pred)

print('Predicted Risk Level:', label[0])
print()
print('Model working perfectly!')